# Exercise 8: Supervised segmentation with a 2D U-Net

In this exercise you will train a supervised U-Net on a set of 2D coronal slices consisting of MR scans and the corresponding tissue segmentations. Below you will find a fully working example of the training code. However, some of the choices, such as the architecture, data augmentation steps, optimizer, etc., are not necessarily optimal. You should make changes and/or add to the example code to improve the validation scores of the architecture. The places in the code were modifications should be made are noted with a TODO-flag.

The exercise will run challenge-style meaning that after your edits, you will run your optimized network on a validation data set as well to see if the performance is the same as on the validation data, i.e., to test if the network generalizes to unseen data, on the next class a podium will be revealed with the groups that performed better

In the report please describe the changes you made to the code, and why you chose those specific changes. Additionally, once the test data is released please comment on the performance of the model on the test data.

>**Tip**:
>
> Don't rely only on metrics! to evaluate your network performance make sure to visualize the data and the segmentations as well!

## Initial steps
To run the code you need to do two things:

1) Install monai

2) Upload the training and validation data so you can train the model

The following two code blocks have the commands to do that.

## Additional tricks and tips

- You can change the hardware the code runs on by selecting (in the top of the page) *Runtime -> Change Runtime Type*. This will allow you to select a CPU (default) or GPU. Selecting GPU will make the code run faster.

- Once you finish a training run, there will be a folder called */log*. You can visualize the file inside that folder using tensorboard by calling

```
%load_ext tensorboard
%tensorboard --logdir log
```

- The pip command might give an error saying that it could not solve all dependencies. This is most likely **not** a problem for us, so just go ahead and try to execute the code.

- **Important:** Whenever you change the *Runtime Type* or *Restart Runtime* everything will be cleared. This means the training data, installed packages, training runs will all be gone and you need to execute the steps from the start. The training code saves the trained model every *nth* epoch. You should download the model to your computer when you're happy with it. Otherwise testing it later will not be possible! (Or you need to at least run the training again)



In [ ]:
# Install monai, you only need to do this the first time the note book is run
!pip install "monai[all]"

In [2]:
# Upload the training data zip file. Click on Browse and upload the file.
# The zip file will show up under the folder icon on the left.
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Unzip the training data
!unzip training_data_exercise.zip

## U-Net training code
The following code block has the U-Net training code. The code bits you should modify have a TODO-statement above them

In [4]:
# This code is modified from the MONAI 2D segmentation tutorial.
# The license below

# Copyright (c) MONAI Consortium
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     http://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import logging
import os
import sys
from glob import glob
import argparse
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.data import create_test_image_2d, list_data_collate, decollate_batch, DataLoader
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import (
    Activations,
    EnsureChannelFirstd,
    AsDiscrete,
    Compose,
    LoadImaged,
    RandSpatialCropd,
    EnsureChannelFirst,
    ToTensord,
)
from monai.visualize import plot_2d_or_3d_image
import requests


def _get_data(train_dir):
    images = sorted(glob(os.path.join(train_dir, "*T1w.npy")))
    segs = sorted(glob(os.path.join(train_dir, "*T1w_tissue_segmentation.npy")))
    files = [{"img": img, "seg": seg} for img, seg in zip(images, segs)]
    return files

def _get_test_data(npz_path="test_data.npz"):
    npz = np.load(npz_path)
    case_ids = list(npz.files)

    test_files = []
    for cid in case_ids:
      img_arr = npz[cid].astype(np.float32)

      test_files.append({
          "img": img_arr,
          "subject_id": cid,
      })

    return test_files


def _get_training_transforms():
    # TODO:
    # These are the basic necessary transforms
    # for loading the training data.
    # The two first are necessary, i.e.,
    # loading and reshaping the data.
    # Add as many as you feel necessary after those two.
    # It's probably smart to keep the cropping as the last step though
    # feel free to change the size. It's used in the validation loop.
    #
    # See available transforms here:
    # https://monai.readthedocs.io/en/1.5.1/transforms.html#
    #
    # NOTE: remember to import whatever you use, you can see examples
    # at the beginning of this code block.
    # NOTE: Not all augmentations are suitable for *both* images and segmentations
    # if you want to only apply an augmentation on the image, you should only pass
    # ["img"] as the keys

    roi_size = [200, 200]

    train_transforms = Compose(
        [
            LoadImaged(keys=["img", "seg"]),
            EnsureChannelFirstd(keys=["img", "seg"]),
            RandSpatialCropd(
                  keys=["img", "seg"],roi_size=roi_size),
        ]
    )


    return train_transforms, roi_size


def _get_validation_transforms():
    # NOTE: you do not need to change the validation transforms
    # only the training transforms.

    val_transforms = Compose(
        [
            LoadImaged(keys=["img", "seg"]),
            EnsureChannelFirstd(keys=["img", "seg"]),
        ]
    )

    return val_transforms

def _get_test_transforms():
  # NOTE: you do not need to change the test transforms
  # only the training transforms.

  return Compose([
        EnsureChannelFirstd(keys=["img"], channel_dim="no_channel"),
        ToTensord(keys=["img"]),
    ])


def _get_data_loaders(train_files, train_transforms, val_files, val_transforms):

    train_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
    train_loader = DataLoader(
        train_ds,
        batch_size=2,
        shuffle=True,
        num_workers=2,
        collate_fn=list_data_collate,
        pin_memory=torch.cuda.is_available(),
    )
    # create a validation data loader
    val_ds = monai.data.Dataset(data=val_files, transform=val_transforms)
    val_loader = DataLoader(val_ds, batch_size=1, num_workers=2, collate_fn=list_data_collate)

    return train_loader, val_loader

def _loss_function(outputs, labels):
    # TODO:
    # I'm only using a Dice loss here.
    # You can change that (or keep it)
    # and also add different losses.
    # If you want to add losses
    # first grab the loss from the monai module
    # (like is done for Dice here)
    # and then add it to the full_loss.
    # Feel free to use different weights.
    #
    # See available segmentation losses here:
    # https://monai.readthedocs.io/en/1.5.1/losses.html
    #

    dice_loss = monai.losses.DiceLoss(to_onehot_y=True, softmax=True)
    full_loss = dice_loss(outputs, labels)

    return full_loss

def _get_model(device):
    # TODO:
    # I'm using the most basic U-Net implemented in MONAI
    # here. You can either modify this architecture or
    # pick another one. The full list is here:
    # https://monai.readthedocs.io/en/1.5.1/networks.html#nets
    #
    # NOTE: The spatial_dims, in_channels and out_channels
    # will not change if you pick another network. Those are
    # fixed by the training data.

    model = monai.networks.nets.BasicUNet(
        spatial_dims=2,
        in_channels=1,
        out_channels=4,
        features=(16, 32, 32, 64, 128, 16),
    ).to(device)

    return model

def _get_optimizer(model):
    # TODO:
    # I'm using vanilla stochastic gradient descent here.
    # You can change it's paremeters, e.g., momentum, or
    # change the optimizers. See available algorithms here:
    # https://pytorch.org/docs/stable/optim.html#algorithms

    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0)

    return optimizer

# Training function start here
def train(train_dir, validation_dir):
    monai.config.print_config()
    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Get the training data files (images and segmentation)
    train_files = _get_data(train_dir)

    # Set up training transforms.
    train_transforms, roi_size = _get_training_transforms()

    # Get the validation data files (images and segmentations)
    val_files = _get_data(validation_dir)

    # Get the validation transforms.
    val_transforms = _get_validation_transforms()


    # Get data loaders for running the training and validation
    train_loader, val_loader = _get_data_loaders(train_files, train_transforms, val_files, val_transforms)

    # The device variable is used to automatically run the training on
    # the GPU if its available, otherwise the CPU is used
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # TODO: Change the _get_model function
    model = _get_model(device)

    # TODO: Change the _get_optimizer function
    optimizer = _get_optimizer(model)


    # Final thing before training. Setup the validation metric so we can measure
    # the validation performance during training.
    # I'm not including the background into the validation dice as we are more interest
    # in the foreground structures.
    dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

    # Define post-processing steps for the output so we can compare it to the validation segmentations
    # The loss does this automatically because we have defined
    # to_onehot_y=True, softmax=True
    # so this is not necessary for training, just for validation
    post_trans = Compose([Activations(softmax=True), AsDiscrete(argmax=True)])

    # Okay start the training loop
    # Feel free to change the number of
    # epochs and the other parameters

    # Validation interval
    val_interval = 10

    # Keep track of the best metric & epoch
    best_metric = -1
    best_metric_epoch = -1

    # TODO:
    # You can use even more the 500 epochs if you want.

    epochs = 500

    # Losses and metrics
    epoch_loss_values = list()
    metric_values = list()

    # Save a log into a directory called log
    writer = SummaryWriter(log_dir='./log')

    for epoch in range(epochs):
        print("-" * 100)
        print(f"epoch {epoch + 1}/{epochs}")
        model.train()
        epoch_loss = 0
        step = 0
        for batch_data in train_loader:
            step += 1

            #Get training data for this batch
            inputs, labels = batch_data["img"].to(device), batch_data["seg"].to(device)
            optimizer.zero_grad()

            # Push the input through the model
            outputs = model(inputs)

            # Get the loss
            loss = _loss_function(outputs, labels)

            # Backpropagate and call the optimizer
            loss.backward()
            optimizer.step()

            # Store the losses
            epoch_loss += loss.item()
            print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")
            writer.add_scalar("train_loss", loss.item(), len(train_loader) * epoch + step)

        # These statements plot examples into the tensorboard log dile
        plot_2d_or_3d_image(inputs[0], epoch + 1, writer, index=0, tag="ínput_image")
        plot_2d_or_3d_image(labels[0], epoch + 1, writer, index=0, tag="input_labeling")
        output_tmp = [post_trans(i) for i in decollate_batch(outputs)]
        plot_2d_or_3d_image(output_tmp[0], epoch + 1, writer, index=0, tag="input_prediction")

        # Save the average loss per epoch
        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

        # Run the validation, every val_interval steps
        if (epoch + 1) % val_interval == 0:
            model.eval()

            # The no_grad is just to tell torch that it doesn't need to backpropagate here
            with torch.no_grad():
                val_images = None
                val_labels = None
                val_outputs = None
                im_num = 0
                for val_data in val_loader:
                    val_images, val_labels = val_data["img"].to(device), val_data["seg"].to(device)
                    sw_batch_size = 4
                    val_outputs = sliding_window_inference(val_images, roi_size, sw_batch_size, model)
                    val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                    # compute metric for current iteration
                    dice_metric(y_pred=val_outputs, y=val_labels)

                    # Plot the validation images, labels and predictions into the log file
                    plot_2d_or_3d_image(val_images, epoch + 1, writer, index=0, tag="validation_image"+str(im_num))
                    plot_2d_or_3d_image(val_labels, epoch + 1, writer, index=0, tag="validation_labeling"+str(im_num))
                    plot_2d_or_3d_image(val_outputs, epoch + 1, writer, index=0, tag="validation_prediction"+str(im_num))
                    im_num += 1

                # aggregate the final mean dice result
                metric = dice_metric.aggregate().item()
                # reset the status for next validation round
                dice_metric.reset()
                metric_values.append(metric)

                # Keep track of the best metric and save the best model
                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    torch.save(model.state_dict(), "best_metric_model_segmentation2d_dict.pth")
                    print("saved new best metric model")
                print(
                    "current epoch: {} current mean dice: {:.4f} best mean dice: {:.4f} at epoch {}".format(
                        epoch + 1, metric, best_metric, best_metric_epoch
                    )
                )
                writer.add_scalar("val_mean_dice", metric, epoch + 1)

    print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
    writer.close()

# This function is used for the testing the performance once the test data
# You don't need to change this
def test(test_dir):
    monai.config.print_config()
    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    test_files = _get_test_data(test_dir)
    test_transforms = _get_test_transforms()

    test_ds = monai.data.Dataset(data=test_files, transform=test_transforms)

    test_loader = DataLoader(test_ds, batch_size=1, num_workers=2, collate_fn=list_data_collate)

    post_trans = Compose([Activations(softmax=True), AsDiscrete(argmax=True, keepdim=False)])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists('output'):
        os.mkdir('output')

    out_path = "output/predictions.npz"

    model = _get_model(device)
    model.load_state_dict(torch.load("best_metric_model_segmentation2d_dict.pth"))

    model.eval()

    roi_size = (200, 200)
    sw_batch_size = 4

    preds_dict = {}

    sub = 0
    with torch.no_grad():
        for batch in test_loader:
            img = batch["img"].to(device)
            subj_id = batch["subject_id"][0]

            logits = sliding_window_inference(img, roi_size, sw_batch_size, model)
            preds_list = [post_trans(i) for i in decollate_batch(logits)]
            pred = preds_list[0]

            pred_np = pred.cpu().numpy()
            pred_np = pred_np.astype(np.uint8)

            preds_dict[str(subj_id)] = pred_np


    np.savez_compressed(out_path, **preds_dict)
    print(f"Saved predictions to: {out_path}")




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Caskroom/miniconda/base/envs/mia/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Caskroom/miniconda/base/envs/mia/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/mia/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/mia/lib/python3.9/site-packag

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## Running the training code
The following code block will run the training code and keep track of it using tensorboard

In [ ]:
# Fire up training
train('./training_data_exercise/training_data/', './training_data_exercise/validation_data/')

In [ ]:
# Visualize using tensorboard
%load_ext tensorboard
# Fire up tensorboard
%tensorboard --logdir log


## Testing the model

Feeling confident about your network ... maybe even convinced it's the **best**!?

Now's the moment to find out.

You can evaluate your model by inserting your group name and press submit, before doing so, please make sure the following steps are completed:

### Before you run the test:

1. Ensure your best model configuration is available in Colab.
You will need the file:
`best_metric_model_segmentation2d_dict.pth`
This file is automatically generated when you run training. If it is not already stored in your Colab environment, upload it using the upload cell at the start of the notebook.

2. Run the U-Net code block first.
The testing script depends on the U-Net implementation and the associated testing utilities imported in that section.

### Submission:
When you press Submit, we will evaluate your model on a brand-new, unseen dataset. Your predicted segmentations will be compared against our ground-truth labels, and your final performance will be scored accordingly.

No worries, you can sumbit as much as you want, but you have to keep the same name

### Results:
Next Tuesday is verdict day: your segmentations meet our ground truths, and your score tells the story.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import requests

def sumbit_results(team_name):
    with open("output/predictions.npz", "rb") as f:
        files = {"file": ("predictions.npz", f, "application/octet-stream")}
        data = {"name": team_name}
        r = requests.post("https://mia-challenge.drcmr.dk/dice-score", files=files, data=data)
    return r

# Text input
team_name_input = widgets.Text(
    description="Team name:",
    placeholder="Enter your team name",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Submit button (starts disabled)
submit_btn = widgets.Button(
    description="Submit",
    button_style="success",
    disabled=True
)

# Output area for feedback
out = widgets.Output()

def validate_name(change=None):
    name = team_name_input.value.strip()
    submit_btn.disabled = (name == "")

team_name_input.observe(validate_name, names="value")


def render_success(payload):
    overall = payload["score"]
    name = payload.get("name", "")
    per_subject = payload["per_subject"]

    # Big general score
    display(HTML(
        f"""
        <div style="font-size:20px; margin-top:8px;">
            ✅ <b>Team:</b> {name}<br>
            🏆 <b>General Dice Score:</b>
            <span style="font-size:28px;">{overall:.4f}</span>
        </div>
        """
    ))

    # Table per subject
    df = pd.DataFrame(
        [{"subject": k, "dice": v} for k, v in per_subject.items()]
    ).sort_values("subject").reset_index(drop=True)

    display(df)

    # Extra quick stats
    best_subj = df.loc[df["dice"].idxmax()]
    worst_subj = df.loc[df["dice"].idxmin()]

    display(HTML(
        f"""
        <div style="margin-top:8px;">
            🌟 <b>Best subject:</b> {best_subj['subject']} ({best_subj['dice']:.4f})<br>
            🥲 <b>Worst subject:</b> {worst_subj['subject']} ({worst_subj['dice']:.4f})
        </div>
        """
    ))


def render_error(payload):
    detail = payload.get("detail", str(payload))
    display(HTML(
        f"""
        <div style="color:#b00020; font-weight:700; font-size:16px; margin-top:8px;">
            ❌ Submission failed: {detail}
        </div>
        """
    ))


def on_submit_clicked(b):
    with out:
        clear_output()
        name = team_name_input.value.strip()
        if not name:
            render_error({"detail": "Please enter a team name."})
            return

        # Disable UI while running
        submit_btn.disabled = True
        team_name_input.disabled = True
        display(HTML("<div>Running inference + submitting…</div>"))

        try:
            # 1) run inference to create output/predictions.npz
            test("test_data.npz")

            clear_output()
            display(HTML("<div>Uploading to server…</div>"))

            # 2) submit to server
            r = sumbit_results(name)

            clear_output()
            if r.status_code == 200:
                render_success(r.json())
            else:
                try:
                    render_error(r.json())
                except Exception:
                    render_error({"detail": r.text})

        except Exception as e:
            clear_output()
            render_error({"detail": repr(e)})

        finally:
            # Re-enable UI
            team_name_input.disabled = False
            validate_name()  # re-enable submit if name still valid


submit_btn.on_click(on_submit_clicked)

display(widgets.VBox([team_name_input, submit_btn, out]))
